# 手撕 AUC

AUC 的定义是 ROC 曲线下的面积。它的物理意义是，随机选择一对正负样本，模型对正样本的打分大于对负样本打分的概率。

法1：利用物理意义，直接遍历正负样本对，由大数定律，用频率近似概率

$AUC = \frac{正>负次数 + 0.5 * 正=负次数}{正样本数 * 负样本数}$

In [ ]:
def auc(labels, preds):

    # 拆分正负样本索引
    pos = [i for i in range(len(labels)) if labels[i] == 1]
    neg = [i for i in range(len(labels)) if labels[i] == 0]

    fz = 0
    fm = len(pos) * len(neg)

    # 统计所有正负样本对中，模型把相对位置排序正确的数量
    for i in pos:
        for j in neg:
            if preds[i] > preds[j]:
                fz += 1
            elif preds[i] == preds[j]:
                fz += 0.5

    auc_value = fz / fm
    return auc_value

# ===== 算例 =====
labels = [1, 1, 0, 1, 0, 0]
preds  = [0.9, 0.8, 0.7, 0.7, 0.7, 0.6]

print("AUC =", auc(labels, preds))

法 2：rank 排序法，基于 Mann-Whitney U 统计量。

$U = rank^+ - \frac{n^+*(n^++1)}{2}$ 

- 第一部分表示正样本秩次和。
- 第二部分表示如果所有正样本都排在最前面，它们天然会占据的最小秩和。减去该部分是为了消除正样本自身占据秩次带来的贡献。

In [ ]:
def auc(labels, preds):

    # 组合并按预测值升序排序
    data = sorted(zip(preds, labels), key=lambda x: x[0])
    
    pos_num = sum(labels)
    neg_num = len(labels) - pos_num
    
    # rank = 预测值升序排序后的下标 + 1
    rank_sum = 0
    i = 0
    while i < len(data):
        j = i
        # 找到所有同分的样本
        while j < len(data) and data[j][0] == data[i][0]:
            j += 1
        
        # 计算平均排名
        # data[i:j] 的下标范围是 i 到 j-1（不包括 data[j])
        # 对应的 rank 是 i+1 到 j，因此平均 rank 是 (i+1+j)/2
        avg_rank = (i + 1 + j) / 2.0
        
        # 只累加正样本的排名
        for k in range(i, j):
            if data[k][1] == 1:
                rank_sum += avg_rank
        i = j
    
    # 套用公式
    auc_value = (rank_sum - pos_num * (pos_num + 1) / 2) / (pos_num * neg_num)
    return auc_value

# ===== 算例 =====
labels = [1, 1, 0, 1, 0, 0]
preds  = [0.9, 0.8, 0.7, 0.7, 0.7, 0.6]

print("AUC =", auc(labels, preds))

AUC = 0.8888888888888888
